# Dataset Inspection and Preprocessing

This notebook loads the captured network traffic dataset, performs basic cleaning, and prepares the data for feature engineering and machine learning analysis. The dataset originates from a simulated maritime Operational Technology (OT) network environment where both normal navigation traffic and attack scenarios were generated.

## Import Required Libraries

In [1]:
# Import libraries used for dataset manipulation and numerical operations
import pandas as pd
import numpy as np

## Load Extracted Packet Dataset

The packet capture (PCAP) file was converted to CSV using Tshark. The resulting dataset contains packet-level information including timestamps, IP addresses, packet size, and inter-arrival times.

In [2]:
# Load the extracted packet dataset produced by tshark
df = pd.read_csv("../data/extracted_csv/dataset_v3.csv")

# Display dataset dimensions (rows, columns)
print(df.shape)
# Show first few rows 
df.head()

(15331, 7)


,frame.time_epoch,ip.src,ip.dst,udp.srcport,udp.dstport,frame.len,frame.time_delta
0,1.774143e+09,192.168.56.10,192.168.56.15,47927,5005,156,0.000000
1,1.774143e+09,192.168.56.10,192.168.56.15,47927,5005,155,1.625913
2,1.774143e+09,192.168.56.10,192.168.56.15,47927,5005,157,1.975084
3,1.774143e+09,192.168.56.10,192.168.56.15,47927,5005,185,1.862526
4,1.774143e+09,192.168.56.10,192.168.56.15,47927,5005,172,2.117311


## Data Cleaning

Initial cleaning removes incomplete rows and converts relevant columns into numeric format for further analysis.

In [3]:
# Remove rows containing missing values
df = df.dropna()

# Convert timestamp column to numeric format for time-based analysis
df["frame.time_epoch"] = pd.to_numeric(df["frame.time_epoch"])
# Convert packet length to numeric for statistical analysis
df["frame.len"] = pd.to_numeric(df["frame.len"])
# Convert inter-arrival time to numeric for timing analysis
df["frame.time_delta"] = pd.to_numeric(df["frame.time_delta"])

## Removal of Non-Relevant Traffic

Infrastructure traffic generated by the virtualisation environment (e.g., host system communication) is removed to ensure the dataset contains only experiment-related network packets.

In [4]:
# Remove packets originating from the VMware host interface
df = df[~df["ip.src"].isin(["192.168.56.1"])]
df = df[~df["ip.dst"].isin(["192.168.56.1"])]

# Remove monitoring node traffic
df = df[df["ip.src"] != "192.168.56.30"]

## Packet Ordering

Packets are sorted chronologically to preserve the correct ordering of network events. This is important for calculating timing-based flow features later in the analysis.

In [5]:
# Sort packets chronologically to preserve network event order
df = df.sort_values("frame.time_epoch")

# Reset index after sorting
df = df.reset_index(drop=True)

## Traffic Labelling

In addition to binary classification, the dataset is labelled for a multiclass classification task. Each attack scenario generated during the experiment is assigned a unique label.

The attack periods were recorded during the experiment and mapped to timestamps within the dataset. The labels used are:

0 – Normal traffic  
1 – Spoofed navigation attack  
2 – Irregular timing attack  
3 – Burst flooding attack

In [6]:
from datetime import datetime

def to_epoch(h, m, s):
    return datetime(2026, 3, 22, h, m, s).timestamp()

### Define Attack Time Windows

The attack start and end times were recorded during the traffic generation phase. These timestamps are converted into epoch time so they can be matched with the packet timestamps contained in the dataset.

In [7]:
# SPOOF
spoof_windows = [
    (to_epoch(1,32,5), to_epoch(1,34,11)),
    (to_epoch(1,47,37), to_epoch(1,49,48)),
    (to_epoch(2,3,16),  to_epoch(2,5,22)),
    (to_epoch(2,19,2),  to_epoch(2,21,9))
]

# TIMING
timing_windows = [
    (to_epoch(1,37,28), to_epoch(1,39,35)),
    (to_epoch(1,53,1),  to_epoch(1,55,9)),
    (to_epoch(2,8,38),  to_epoch(2,10,46)),
    (to_epoch(2,24,26), to_epoch(2,26,32))
]

# FLOOD
flood_windows = [
    (to_epoch(1,42,47), to_epoch(1,44,22)),
    (to_epoch(1,58,25), to_epoch(2,0,0)),
    (to_epoch(2,14,0),  to_epoch(2,15,36)),
    (to_epoch(2,29,43), to_epoch(2,31,24))
]

### Apply Multiclass Labels

Packets that fall within the defined attack windows are assigned the corresponding attack label. All packets outside these windows remain labelled as normal traffic (0).

In [8]:
# Default = normal
df["multi_label"] = 0

# Spoof
for start, end in spoof_windows:
    df.loc[(df["frame.time_epoch"] >= start) &
           (df["frame.time_epoch"] <= end), "multi_label"] = 1

# Timing
for start, end in timing_windows:
    df.loc[(df["frame.time_epoch"] >= start) &
           (df["frame.time_epoch"] <= end), "multi_label"] = 2

# Flood
for start, end in flood_windows:
    df.loc[(df["frame.time_epoch"] >= start) &
           (df["frame.time_epoch"] <= end), "multi_label"] = 3

### Binary Classification Labels

Binary labels are derived from the multiclass labels to distinguish between normal and malicious traffic.

0 = Normal
1 = Attack

In [9]:
df["binary_label"] = df["multi_label"].apply(lambda x: 0 if x == 0 else 1)

### Dataset Distribution Check

The distribution of labels is examined to confirm that all traffic classes are represented.

In [10]:
print("Binary distribution:")
print(df["binary_label"].value_counts())

print("\nMulticlass distribution:")
print(df["multi_label"].value_counts())

Binary distribution:
binary_label
1    14092
0     1239
Name: count, dtype: int64

Multiclass distribution:
multi_label
3    13527
0     1239
1      301
2      264
Name: count, dtype: int64


## Processed Dataset Output

The cleaned and labelled dataset is saved for use in the feature engineering stage of the machine learning pipeline.

In [11]:
# Save processed dataset to data directory
df.to_csv("../data/processed/dataset_v3_labeled.csv", index=False)